<a href="https://colab.research.google.com/github/y-ooe/googleColab-test/blob/main/GPT_SoVITS_Colab_v2Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS on Colab（v2Pro / 日本語）少量の音声からその人の声で日本語を喋らせるモデルを作ります。公式ノートブックは依存関係で複数つまずくため、修正済みの版です。**上から1セルずつ実行。** 事前に ランタイム → ランタイムのタイプを変更 → **T4 GPU**。- セル2の直後にランタイムが再起動します（正常）- セットアップは10〜20分- 切断されると環境は消滅。セル1からやり直し（Driveに保存した学習結果は残る）

In [ ]:
# RAMの使用率を見れる関数(不必要)
# なんか無料枠は12GBしかRAMがないらしく、よくクラッシュするので以下の関数とかを使いエラーが起きる場所を確認するといいかも(大江)
# 参考サイト：https://qiita.com/sakaimaging/items/04ec48cb20baf4ce2b47
import os
import psutil

def print_memory_status():
    virtual_mem = psutil.virtual_memory()
    print(f"Total RAM     : {virtual_mem.total / 1024**3:.2f} GB")
    print(f"Available RAM : {virtual_mem.available / 1024**3:.2f} GB")
    print(f"Used RAM      : {(virtual_mem.total - virtual_mem.available) / 1024**3:.2f} GB")
    print(f"Memory usage %: {virtual_mem.percent} %")

    process = psutil.Process(os.getpid())
    print(f"Current process memory usage: {process.memory_info().rss / 1024**3:.2f} GB")

print_memory_status()


Total RAM     : 12.67 GB
Available RAM : 11.69 GB
Used RAM      : 0.98 GB
Memory usage %: 7.8 %
Current process memory usage: 0.10 GB


## 1. GPU確認

In [ ]:
# Tesla T4 等(N/Aの上の行)が表示されればOK。出なければランタイムのタイプを変更する
!nvidia-smi

Tue Sep 15 04:49:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. conda導入> ⚠️ **このセルだけ単独で実行。** 導入後にカーネルが再起動するため、> 同じセルに他の処理を書くと実行されません。「クラッシュしました」の表示は正常です。

In [ ]:
!pip install -q condacolab
import condacolab
# 実行後にカーネルが自動再起動する（正常）。再実行せず次のセルへ
condacolab.install_from_url(    "https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:01:47
🔁 Restarting kernel...


## 3. 再起動後の確認`✨🍰✨ Everything looks OK!` が出ればOK。

In [ ]:
import condacolab
condacolab.check()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


## 4. Driveのマウント（学習に進むなら必須）Colabが切れると `/content` は消えます。動作確認だけならスキップ可。

In [ ]:
# 6.のセットアップがうまくいけばもう一度このセルを実行する(大江)
# ↑毎回pip install のデータとかが消えるため


from google.colab import drive
import os
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/gpt_sovits'
# 保存先。変更可
os.makedirs(SAVE_DIR, exist_ok=True)
print('保存先:', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
保存先: /content/drive/MyDrive/gpt_sovits


## 5. セットアップスクリプトの書き出しまだ実行はされません（実行は次のセル）。> ⚠️ このスクリプトは `/content/GPT-SoVITS` を**削除してから**クローンします。> 学習を始めたあとは、必ずセル11でDriveに退避してから実行してください。### <br>公式の install.sh からの変更点| 変更 | 理由 ||---|---|| `opencc` を除外し純Python版に差し替え | C++ビルドに失敗する。日本語では使わない機能 || `libstdcxx-ng` を更新 | pyopenjtalk が `GLIBCXX_3.4.32` を要求 || torch / torchaudio を cu126 で明示 | 放置すると `libcudart.so.13` 不足で落ちる || `markupsafe` を2系に固定 | torchの依存で3系に上がり Gradio と衝突 || `starlette` を1.0未満に固定 | 1.0 で Gradio の画面描画が落ちる |

In [ ]:
%%writefile /content/setup.sh

#!/bin/bash
# GPT-SoVITS セットアップ（Colab / CUDA 12.6 / 日本語）
# 失敗した時点で即終了する。途中で止まったら出力の末尾がエラーの本体

set -e
cd /content

# --- リポジトリ取得 ---
# 中途半端な状態が残っていると clone が失敗するため、毎回消してから取得する

rm -rf GPT-SoVITS
git clone https://github.com/RVC-Boss/GPT-SoVITS.git
cd GPT-SoVITS

# opencc（中国語の繁簡変換）を依存から除外。ビルドが通らず、日本語では不要

sed -i '/^opencc/d' requirements.txt

# --- conda環境 ---

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
# 既にあるので作成しない
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS

# pyopenjtalk が要求する GLIBCXX_3.4.32 のため。pipより先に実行する

conda install -y -c conda-forge libstdcxx-ng
pip install ipykernel
pip install opencc-python-reimplemented

# 上で外した opencc の代替

# --- 本体 ---
# --download-uvr5 は付けていない（ボーカル分離用で数GB。クリーンな録音なら不要）

bash install.sh --device CU126 --source HF

# --- バージョン固定 ---
# index-url の指定が必須。無いと PyPI から CUDA13版が降ってくる

pip install --force-reinstall torch==2.14.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu126

# ⚠️ 順序依存: torch が markupsafe を3系に上げるので、必ずその後で戻す
pip install "markupsafe<3.0"
pip install "starlette<1.0.0"
echo "=== setup done ==="

Writing /content/setup.sh


## 6. セットアップ実行（10〜20分）最後に `=== setup done ===` が出れば成功。止まった場合は `set -e` により失敗地点で停止しているので、**出力の末尾**を見てください。

In [ ]:
!cd /content && bash setup.sh

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /usr/local/lib/libtinfo.so.6: no version information available (required by bash)
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5974, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 5974 (delta 3), reused 0 (delta 0), pack-reused 5954 (from 3)
Receiving objects: 100% (5974/5974), 14.27 MiB | 18.10 MiB/s, done.
Resolving deltas: 100% (3408/3408), done.
Channels:
 - defaults
Platform: linux-64
Solving environment: \ | done

## Package Plan ##

  environment location: /usr/local/envs/GPTSoVITS

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    ca-certificates-2026.8.13  |       h06a4308_0         107 

## 7. 検証`--- すべてOK ---` が出れば次へ。

In [ ]:
# 準備ファイル(検証は次のセル)
%%writefile /content/check_env.py
import torch, torchaudio, gradio, starlette, markupsafe

print('torch      ', torch.__version__)       # 2.14.0+cu126
print('torchaudio ', torchaudio.__version__)  # 2.11.0+cu126
print('gradio     ', gradio.__version__)
print('starlette  ', starlette.__version__)   # 1.0未満
print('markupsafe ', markupsafe.__version__)  # 2.x
print('CUDA使用可 ', torch.cuda.is_available())

# CUDA版がズレていると推論時に libcudart で落ちるので先に弾く
assert '+cu126' in torch.__version__,      'torch の CUDA 版が違う'
assert '+cu126' in torchaudio.__version__, 'torchaudio の CUDA 版が違う'
assert torch.cuda.is_available(), 'GPU が見えていない'

print('--- すべてOK ---')

Writing /content/check_env.py


In [14]:
!source activate GPTSoVITS && python /content/check_env.py

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
torch       2.14.0+cu126
torchaudio  2.11.0+cu126
gradio      4.44.1
starlette   0.52.1
markupsafe  2.1.5
CUDA使用可  True
--- すべてOK ---


## 8. 動作したバージョンを記録（推奨）依存パッケージは日によって降ってくる版が変わります。将来壊れたときの比較用。

In [ ]:
import os, shutil
!source activate GPTSoVITS && pip list --format=freeze > /content/working_versions.txtif os.path.exists('/content/drive/MyDrive'):    os.makedirs('/content/drive/MyDrive/gpt_sovits', exist_ok=True)    shutil.copy('/content/working_versions.txt',                '/content/drive/MyDrive/gpt_sovits/working_versions.txt')    print('Driveに保存しました')else:    print('Drive未マウント（セル4で保存されます）')

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `source activate GPTSoVITS && pip list --format=freeze > /content/working_versions.txtif os.path.exists('/content/drive/MyDrive'):    os.makedirs('/content/drive/MyDrive/gpt_sovits', exist_ok=True)    shutil.copy('/content/working_versions.txt',                '/content/drive/MyDrive/gpt_sovits/working_versions.txt')    print('Driveに保存しました')else:    print('Drive未マウント（セル4で保存されます）')'


## 9. WebUI起動> ⚠️ **実行しっぱなしにする。** 停止するとWebUIも落ちます。出力の `https://xxxx.gradio.live` を開いてください。`http://0.0.0.0:9874` はColab内部のアドレスなので開けません。

In [ ]:
# is_share=True … 公開URL(*.gradio.live)を発行。Colabでは必須
# LD_PRELOAD   … システム側の新しい libstdc++ を優先読み込み。日本語合成に必要
# ja_JP        … 画面を日本語表示に（省略すると中国語）
!cd /content/GPT-SoVITS && source activate GPTSoVITS && export is_share=True && LD_PRELOAD=/usr/lib/x86_64-linux-gnu/libstdc++.so.6 python webui.py ja_JP

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://ca5c524f64d16284d5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
"/usr/local/envs/GPTSoVITS/bin/python" -s GPT_SoVITS/inference_webui.py "ja_JP"
CUDA Graph: support check passed, auto-enabled
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/content/GPT-SoVITS/GPT_SoVITS/inference_webui.py", line 198, in <module>
    bert_model = bert_model.half().to(device)
  File "/usr/local/envs/GPTSoVITS/lib/python3.10/site-packages/transformers/modeling_utils.py", line 4343, in to
    return super().to(*args, **kwargs)
  File "/usr/local/envs/GPTSoVITS/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1388, in to
    return self._appl

## 10. ポート転送（公開URLが出ないとき）子プロセスとして起動する推論UIは公開URLが出ないことがあります。**セル9を動かしたまま**実行してください。

In [ ]:
from google.colab.output import eval_js
# 9874:メインWebUI  9872:TTS推論UI  9873:ボイスチェンジャー等
# 未起動のポートのURLを開いてもエラーになる（正常）for port in (9874, 9872, 9873):
print(f'{port}: {eval_js(f"google.colab.kernel.proxyPort({port})")}')

## 11. 学習結果をDriveに退避学習が終わったら**必ず**実行。`SoVITS_weights*` が声質、`GPT_weights*` が抑揚のモデルです。

In [ ]:
import os, shutil, glob
SAVE_DIR = '/content/drive/MyDrive/gpt_sovits'
assert os.path.exists('/content/drive/MyDrive'), 'Driveをマウントしてください（セル4）'
os.makedirs(SAVE_DIR, exist_ok=True)
# バージョンにより接尾辞が付く（例: SoVITS_weights_v2Pro）のでワイルドカードで拾う
for pattern in ['SoVITS_weights*', 'GPT_weights*']:
  for src in glob.glob(f'/content/GPT-SoVITS/{pattern}'):
    dst = os.path.join(SAVE_DIR, os.path.basename(src))
    if os.path.isdir(src):
      shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
      shutil.copy(src, dst)
print('保存:', dst)print('--- 完了 ---')

---## WebUI側の作業手順| 順 | タブ | 内容 ||---|---|---|| 1 | 1C-推論 | ベースモデルでゼロショット合成。動作確認用 || 2 | 0b | 音声を3〜10秒に分割 || 3 | 0c | ASRで文字起こし（Faster Whisper、言語は `ja`） || 4 | 0d | 書き起こしを手動校正 ← **品質を最も左右する。飛ばさない** || 5 | 1A | 前処理。3つのボタンを**上から1つずつ**（同時押しはVRAM不足） || 6 | 1B | 学習。SoVITS 8〜15 epoch / GPT 10〜20 epoch || 7 | 1C | 学習したモデルで推論。GPTとSoVITSのバージョンは揃える || 8 | セル11 | Driveへ退避 |**推論時の入力**: 参照音声は3〜10秒厳守。参照テキストは音声の内容を一字一句正確に。言語は両方「日文」。**収録のコツ**: ITAコーパスの台本を使うと音素バランスが設計済みで効率が良い。48kHz / 無圧縮 / 静かな環境。チャット用途なら感情バリエーションを含める。## トラブルシューティング| 症状 | 対処 ||---|---|| `ModuleNotFoundError` | 環境違いかインストール失敗。`source activate GPTSoVITS` の有無を確認 || `libXXX.so.数字` が無い | 数字＝要求バージョン。CUDA/C++の版ズレ || `unhashable type: 'dict'` | `pip install "starlette<1.0.0"` || `GLIBCXX_...` not found | libstdc++ が古い。`LD_PRELOAD` を確認 || `libcudart.so.13` が無い | torchaudio がCUDA13版。cu126のindex-urlで入れ直す || pipの赤い警告 | `Successfully installed` があれば成功。衝突分だけ個別に直す || `destination path already exists` | clone失敗で全体停止。`rm -rf` して再実行 || `gradio.live` が出ない | セル10のポート転送を使う |**トレースバックは下から読む。** 最下行が実際のエラーです。